In [2]:
import altair as alt
import pandas as pd
import geopandas as gpd # Requires geopandas -- e.g.: conda install -c conda-forge geopandas
alt.data_transformers.enable('json') # Let Altair/Vega-Lite work with large data sets

pass

In [3]:
names = pd.read_csv("Names_hints/dpt2020.csv", sep=";")
names.drop(names[names.preusuel == '_PRENOMS_RARES'].index, inplace=True)
names.drop(names[names.dpt == 'XX'].index, inplace=True)

names.sample(5)

,sexe,preusuel,annais,dpt,nombre
2642793,2,JANE,1920,30,6
3197641,2,MÉLANIE,1905,75,13
3457790,2,ROLANDE,1919,72,7
2256446,2,DEBRA,1960,17,3
1069122,1,MAEL,2004,51,29


In [4]:
depts = gpd.read_file('Names_hints/departements-version-simplifiee.geojson')

depts.sample(5)

# Keep a reference around to the plain pandas dataframe, without geometry data, just in case
just_names = names

names = depts.merge(names, how='right', left_on='code', right_on='dpt')

names.sample(5)


grouped = (
    names.groupby(['dpt', 'preusuel', 'sexe'], as_index=False)
         .agg({'nombre': 'sum'})
)
grouped = depts.merge(grouped, how='right', left_on='code', right_on='dpt') # Add geometry data back in
grouped

,code,nom,geometry,dpt,preusuel,sexe,nombre
0,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,AARON,1,160
1,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABBY,2,3
2,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABDALLAH,1,7
3,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABDEL,1,3
4,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABDELKADER,1,3
...,...,...,...,...,...,...,...
239574,NaN,NaN,None,974,ÉSAÏE,1,3
239575,NaN,NaN,None,974,ÉTHAN,1,53
239576,NaN,NaN,None,974,ÉTIENNE,1,3
239577,NaN,NaN,None,974,ÉVA,2,32


In [5]:

# Top 10 prénoms les plus donnés
top_names = (
    names.groupby('preusuel')['nombre']
         .sum()
         .nlargest(10)
         .index
)

time_data = (
    names[names.preusuel.isin(top_names)]
    .groupby(['annais', 'preusuel'], as_index=False)
    ['nombre']
    .sum()
)

alt.Chart(time_data).mark_line().encode(
    x=alt.X('annais:O', title='Année'),
    y=alt.Y('nombre:Q', title='Naissances'),
    color='preusuel:N',
    tooltip=['annais', 'preusuel', 'nombre']
).properties(
    width=1200,
    height=600,
    title='Évolution des prénoms les plus populaires'
).interactive()

alt.Chart(...)

In [6]:
import altair as alt
import pandas as pd

# Top 10 prénoms les plus donnés
top_names = (
    names.groupby('preusuel')['nombre']
         .sum()
         .nlargest(10)
         .index
)

time_data = (
    names[names.preusuel.isin(top_names)]
    .groupby(['annais', 'preusuel'], as_index=False)['nombre']
    .sum()
)

# Calcul du pourcentage par année
totaux_annuels = (
    time_data.groupby('annais', as_index=False)['nombre']
             .sum()
             .rename(columns={'nombre': 'total_annuel'})
)

time_data = time_data.merge(totaux_annuels, on='annais')
time_data['pourcentage'] = (
    100 * time_data['nombre'] / time_data['total_annuel']
)

# Bouton radio Nombre / Pourcentage
mode = alt.param(
    name='Mode',
    value='Nombre',
    bind=alt.binding_radio(
        options=['Nombre', 'Pourcentage'],
        name='Affichage : '
    )
)

chart = (
    alt.Chart(time_data)
    .add_params(mode)
    .transform_calculate(
        valeur="""
        Mode == 'Nombre'
        ? datum.nombre
        : datum.pourcentage
        """
    )
    .mark_line()
    .encode(
        x=alt.X('annais:O', title='Année'),
        y=alt.Y(
            'valeur:Q',
            title='Naissances / Pourcentage'
        ),
        color=alt.Color('preusuel:N', title='Prénom'),
        tooltip=[
            alt.Tooltip('annais:O', title='Année'),
            alt.Tooltip('preusuel:N', title='Prénom'),
            alt.Tooltip('nombre:Q', title='Naissances'),
            alt.Tooltip('pourcentage:Q', title='Pourcentage', format='.2f')
        ]
    )
    .properties(
        width=1200,
        height=600,
        title='Évolution des prénoms les plus populaires'
    )
    .interactive()
)

chart

alt.Chart(...)

In [7]:
# Sélecteur de prénom
prenom = alt.param(
    name='Prenom',
    value='Tous',
    bind=alt.binding_select(
        options=['Tous'] + top_names,
        name='Prénom : '
    )
)

chart = (
    alt.Chart(time_data)
    .add_params(mode, prenom)
    .transform_filter(
        "(Prenom == 'Tous') || (datum.preusuel == Prenom)"
    )
    .transform_calculate(
        valeur="""
        Mode == 'Nombre'
        ? datum.nombre
        : datum.pourcentage
        """
    )
    .mark_line(point=True)
    .encode(
        x=alt.X('annais:O', title='Année'),
        y=alt.Y(
            'valeur:Q',
            title='Naissances / Pourcentage'
        ),
        color=alt.Color('preusuel:N', title='Prénom'),
        tooltip=[
            'annais:O',
            'preusuel:N',
            alt.Tooltip('nombre:Q', title='Naissances'),
            alt.Tooltip('pourcentage:Q', title='Pourcentage', format='.2f')
        ]
    )
    .properties(
        width=1200,
        height=600,
        title='Évolution des prénoms'
    )
    .interactive()
)

chart

alt.Chart(...)

In [8]:
import pandas as pd
import numpy as np
import altair as alt

# ==========================================================
# 1. Detect trend-driven names
# ==========================================================

all_names = (
    names.groupby(["annais", "preusuel"], as_index=False)["nombre"]
         .sum()
)

scores = []

MIN_TOTAL = 3000

for first_name, g in all_names.groupby("preusuel"):

    total = g["nombre"].sum()

    if total < MIN_TOTAL:
        continue

    g = g.sort_values("annais").reset_index(drop=True)

    peak_pos = g["nombre"].idxmax()
    peak = g.loc[peak_pos, "nombre"]

    before = g.loc[:peak_pos-1, "nombre"]
    after = g.loc[peak_pos+1:, "nombre"]

    if len(before) < 5 or len(after) < 5:
        continue

    avg_before = before.mean()
    avg_after = after.mean()

    peak_width = (g["nombre"] >= 0.5 * peak).sum()

    trend_score = (
        (peak / (avg_before + 1))
        * (peak / (avg_after + 1))
        / peak_width
    )

    scores.append({
        "preusuel": first_name,
        "trend_score": trend_score,
        "peak": peak,
        "peak_width": peak_width,
        "total": total
    })

trend_scores = (
    pd.DataFrame(scores)
    .sort_values("trend_score", ascending=False)
    .reset_index(drop=True)
)

TOP_TREND = 10

trend_names = (
    trend_scores
    .head(TOP_TREND)["preusuel"]
    .tolist()
)

print("Names detected as trend-driven:")
print(trend_names)

# ==========================================================
# 2. Top historical names
# ==========================================================

TOP_POP = 10

top_names = (
    names.groupby("preusuel")["nombre"]
         .sum()
         .nlargest(TOP_POP)
         .index
         .tolist()
)

# Union of both sets
displayed_names = sorted(
    set(top_names) | set(trend_names)
)

# ==========================================================
# 3. Chart data
# ==========================================================

time_data = (
    names[names["preusuel"].isin(displayed_names)]
    .groupby(["annais", "preusuel"], as_index=False)["nombre"]
    .sum()
)

# Percentages
yearly_totals = (
    time_data.groupby("annais", as_index=False)["nombre"]
             .sum()
             .rename(columns={"nombre": "yearly_total"})
)

time_data = time_data.merge(
    yearly_totals,
    on="annais"
)

time_data["percentage"] = (
    100 * time_data["nombre"]
    / time_data["yearly_total"]
)

# Trend effect flag
time_data["trend_effect"] = (
    time_data["preusuel"].isin(trend_names)
)

# ==========================================================
# 4. Interactive controls
# ==========================================================

display_mode = alt.param(
    name="DisplayMode",
    value="Count",
    bind=alt.binding_radio(
        options=["Count", "Percentage"],
        name="Display: "
    )
)

trend_filter = alt.param(
    name="TrendFilter",
    value="All",
    bind=alt.binding_radio(
        options=["All", "Trend-driven"],
        name="Type: "
    )
)

first_name = alt.param(
    name="FirstName",
    value="All",
    bind=alt.binding_select(
        options=["All"] + displayed_names,
        name="Name: "
    )
)

# ==========================================================
# 5. Chart
# ==========================================================

chart = (
    alt.Chart(time_data)

    .add_params(
        display_mode,
        trend_filter,
        first_name
    )

    # Name filter
    .transform_filter(
        "(FirstName == 'All') || (datum.preusuel == FirstName)"
    )

    # Trend effect filter
    .transform_filter(
        "(TrendFilter == 'All') || datum.trend_effect"
    )

    # Count / Percentage
    .transform_calculate(
        value="""
        DisplayMode == 'Count'
        ? datum.nombre
        : datum.percentage
        """
    )

    .mark_line(point=True)

    .encode(
        x=alt.X(
            "annais:O",
            title="Year"
        ),

        y=alt.Y(
            "value:Q",
            title="Births / Percentage"
        ),

        color=alt.Color(
            "preusuel:N",
            title="Name"
        ),

        tooltip=[
            alt.Tooltip("annais:O", title="Year"),
            alt.Tooltip("preusuel:N", title="Name"),
            alt.Tooltip("nombre:Q", title="Births"),
            alt.Tooltip(
                "percentage:Q",
                title="Percentage",
                format=".2f"
            )
        ]
    )

    .properties(
        width=1200,
        height=600,
        title="Evolution of First Names"
    )

    .interactive()
)

chart

Names detected as trend-driven:
['PAMELA', 'AMANDA', 'STÉPHANIE', 'SEVERINE', 'LAURYNE', 'MARIELLE', 'WILFRID', 'MANDY', 'NOE', 'BRIGITTE']


alt.Chart(...)

Historical leaders → Quels sont les prénoms les plus populaires de l'histoire ?

Trend-driven → Quels prénoms ont connu un effet de mode ?

Growing → Quels prénoms gagnent durablement en popularité ?

Declining → Quels prénoms sont en déclin ?

Stable → Quels prénoms restent stables à travers les générations ?

In [23]:
import pandas as pd
import numpy as np
import altair as alt

# ==========================================================
# 0. CLEAN DATA
# ==========================================================

names = names.copy()

names = names[names["annais"] != "XXXX"]

names["annais"] = pd.to_numeric(
    names["annais"],
    errors="coerce"
)

names = names.dropna(subset=["annais"])

names["annais"] = names["annais"].astype(int)

# ==========================================================
# PARAMETERS
# ==========================================================

MIN_TOTAL = 1000
TOP_N = 6

# ==========================================================
# AGGREGATE DATA
# ==========================================================

all_names = (
    names.groupby(
        ["annais", "preusuel"],
        as_index=False
    )["nombre"]
    .sum()
)

# ==========================================================
# COMPUTE SCORES
# ==========================================================

trend_scores = []
growth_scores = []
decline_scores = []
stability_scores = []

for first_name, g in all_names.groupby("preusuel"):

    total = g["nombre"].sum()

    if total < MIN_TOTAL:
        continue

    g = (
        g.sort_values("annais")
         .reset_index(drop=True)
    )

    peak_pos = g["nombre"].idxmax()

    if 5 <= peak_pos <= len(g) - 6:

        peak = g.loc[peak_pos, "nombre"]

        before = g.loc[:peak_pos - 1, "nombre"]
        after = g.loc[peak_pos + 1:, "nombre"]

        avg_before = before.mean()
        avg_after = after.mean()

        peak_width = (
            g["nombre"] >= 0.5 * peak
        ).sum()

        trend_score = (
            (peak / (avg_before + 1))
            * (peak / (avg_after + 1))
            / peak_width
        )

        trend_scores.append({
            "preusuel": first_name,
            "score": trend_score
        })

    early_period = g[
        g["annais"].between(1950, 1980)
    ]

    late_period = g[
        g["annais"] >= 2010
    ]

    if (
        len(early_period) > 0
        and len(late_period) > 0
    ):

        early = early_period["nombre"].mean()
        late = late_period["nombre"].mean()

        growth_scores.append({
            "preusuel": first_name,
            "score": late / (early + 1)
        })

        decline_scores.append({
            "preusuel": first_name,
            "score": early / (late + 1)
        })

    mean_val = g["nombre"].mean()

    if mean_val > 0:

        cv = (
            g["nombre"].std()
            / mean_val
        )

        stability_scores.append({
            "preusuel": first_name,
            "score": cv
        })

# ==========================================================
# TOP NAMES
# ==========================================================

trend_names = (
    pd.DataFrame(trend_scores)
      .sort_values("score", ascending=False)
      .head(TOP_N)["preusuel"]
      .tolist()
)

growing_names = (
    pd.DataFrame(growth_scores)
      .sort_values("score", ascending=False)
      .head(TOP_N)["preusuel"]
      .tolist()
)

declining_names = (
    pd.DataFrame(decline_scores)
      .sort_values("score", ascending=False)
      .head(TOP_N)["preusuel"]
      .tolist()
)

stable_names = (
    pd.DataFrame(stability_scores)
      .sort_values("score", ascending=True)
      .head(TOP_N)["preusuel"]
      .tolist()
)

historical_names = (
    names.groupby("preusuel")["nombre"]
         .sum()
         .nlargest(TOP_N)
         .index
         .tolist()
)

# ==========================================================
# CUSTOM EVENT-RELATED NAMES
# ==========================================================

event_related_names = [
    "ZINEDINE",    # Coupe du Monde 1998
    "KYLIAN",      # Mbappé
    "SIMONE",      # Simone Veil
    "DIANA",       # Lady Diana
    "CHARLES",     # Roi Charles
]


available_names = set(names["preusuel"].unique())

event_related_names = [
    n for n in event_related_names
    if n in available_names
]


# ==========================================================
# DISPLAYED NAMES
# ==========================================================

displayed_names = sorted(
    set(historical_names)
    | set(trend_names)
    | set(growing_names)
    | set(declining_names)
    | set(stable_names)
    | set(event_related_names)
)

# ==========================================================
# TIME SERIES
# ==========================================================

base_time_data = (
    names[
        names["preusuel"].isin(
            displayed_names
        )
    ]
    .groupby(
        ["annais", "preusuel"],
        as_index=False
    )["nombre"]
    .sum()
)

# ==========================================================
# PERCENTAGES
# ==========================================================

year_totals = (
    names.groupby(
        "annais",
        as_index=False
    )["nombre"]
    .sum()
    .rename(
        columns={
            "nombre": "year_total"
        }
    )
)

base_time_data = (
    base_time_data.merge(
        year_totals,
        on="annais"
    )
)

base_time_data["percentage"] = (
    100
    * base_time_data["nombre"]
    / base_time_data["year_total"]
)

# ==========================================================
# CATEGORY DATASET
# ==========================================================

category_sets = {
    "Historical leaders": historical_names,
    "Trend-driven": trend_names,
    "Growing": growing_names,
    "Declining": declining_names,
    "Stable": stable_names,
    "Event-related": event_related_names
}

frames = []

for category, selected_names in category_sets.items():

    subset = base_time_data[
        base_time_data["preusuel"]
        .isin(selected_names)
    ].copy()

    subset["analysis_category"] = category

    frames.append(subset)

time_data = pd.concat(
    frames,
    ignore_index=True
)

# ==========================================================
# HISTORICAL PERIODS
# ==========================================================

periods = pd.DataFrame([
    (1914, 1918, "WWI"),
    (1939, 1945, "WWII"),
    (1946, 1973, "Baby Boom"),
    (1968, 1968, "May 68"),
    (1998, 1998, "World Cup"),
], columns=["start", "end", "period"])

# ==========================================================
# PERIOD BACKGROUNDS
# ==========================================================

period_bands = (
    alt.Chart(periods)
    .mark_rect(opacity=0.12)
    .encode(
        x="start:Q",
        x2="end:Q",
        color=alt.Color("period:N", legend=None)  # ✅ Désactive la légende
    )
)

# ==========================================================
# CONTROLS
# ==========================================================

analysis_mode = alt.param(
    name="AnalysisMode",
    value="Historical leaders",
    bind=alt.binding_radio(
        options=list(category_sets.keys()),
        name="Question: "
    )
)

display_mode = alt.param(
    name="DisplayMode",
    value="Count",
    bind=alt.binding_radio(
        options=[
            "Count",
            "Percentage"
        ],
        name="Display: "
    )
)

selected_name = alt.param(
    name="SelectedName",
    value="All",
    bind=alt.binding_select(
        options=["All"] + displayed_names,
        name="Name: "
    )
)

# ==========================================================
# MAIN CHART
# ==========================================================

main_chart = (
    alt.Chart(time_data)

    .add_params(
        analysis_mode,
        display_mode,
        selected_name
    )

    .transform_filter(
        "datum.analysis_category == AnalysisMode"
    )

    .transform_filter(
        "(SelectedName == 'All') || (datum.preusuel == SelectedName)"
    )

    .transform_calculate(
        value="""
        DisplayMode == 'Count'
        ? datum.nombre
        : datum.percentage
        """
    )

    .mark_line(
        interpolate="monotone",
        strokeWidth=3
    )

    .encode(

        x=alt.X(
            "annais:Q",
            title="Year",
            scale=alt.Scale(domain=[1900, 2020]),
            axis=alt.Axis(
                values=list(range(1900, 2021, 10)),
                labelAngle=0
            )
        ),

        y=alt.Y(
            "value:Q",
            title="Births / Share (%)"
        ),

        color=alt.Color(
            "preusuel:N",
            title="First name",
            scale=alt.Scale(
                scheme="tableau20"
            )
        ),

        tooltip=[
            alt.Tooltip(
                "preusuel:N",
                title="First name"
            ),
            alt.Tooltip(
                "annais:Q",
                title="Year"
            ),
            alt.Tooltip(
                "nombre:Q",
                title="Births",
                format=","
            ),
            alt.Tooltip(
                "percentage:Q",
                title="Share (%)",
                format=".3f"
            ),
            alt.Tooltip(
                "analysis_category:N",
                title="Question"
            )
        ]
    )
)

# ==========================================================
# POINTS
# ==========================================================

points = (
    alt.Chart(time_data)

    .add_params(
        analysis_mode,
        display_mode,
        selected_name
    )

    .transform_filter(
        "datum.analysis_category == AnalysisMode"
    )

    .transform_filter(
        "(SelectedName == 'All') || (datum.preusuel == SelectedName)"
    )

    .transform_calculate(
        value="""
        DisplayMode == 'Count'
        ? datum.nombre
        : datum.percentage
        """
    )

    .mark_circle(
        size=25,
        opacity=0.5
    )

    .encode(
        x="annais:Q",
        y="value:Q",
        color="preusuel:N"
    )
)

# ==========================================================
# EVENT LINES
# ==========================================================
events = pd.DataFrame([ (1914, "WWI"), 
                       (1918, "Armistice"), 
                       (1939, "WWII"), 
                       (1945, "Liberation"), 
                       (1968, "May 68"), 
                       (1998, "World Cup"), 
                       ], columns=["year", "event"])
event_lines = (
    alt.Chart(events)
    .mark_rule(
        color="#8B0000",
        strokeDash=[6, 4],
        opacity=0.5
    )
    .encode(
        x="year:Q"
    )
)

# ==========================================================
# EVENT LABELS
# ==========================================================

event_labels = (
    alt.Chart(events)
    .mark_text(
        align="left",
        baseline="top",
        dx=4,
        fontSize=11,
        color="#8B0000"
    )
    .encode(
        x="year:Q",
        y=alt.value(5),
        text="event:N"
    )
)

# ==========================================================
# PERIOD LABELS
# ==========================================================

period_labels = (
    alt.Chart(periods)
    .transform_calculate(
        middle="(datum.start + datum.end)/2"
    )
    .mark_text(
        dy=-320,
        fontSize=12,
        fontWeight="bold",
        color="black"
    )
    .encode(
        x="middle:Q",
        text="period:N"
    )
)

# ==========================================================
# LABELS AT END OF LINES
# ==========================================================

labels_data = (
    time_data.loc[
        time_data.groupby(
            ["analysis_category", "preusuel"]
        )["nombre"].idxmax()
    ]
    .copy()
)
# ==========================================================
# FINAL CHART
# ==========================================================
line_labels = (
    alt.Chart(labels_data)

    .add_params(
        analysis_mode,
        display_mode,
        selected_name
    )

    .transform_filter(
        "datum.analysis_category == AnalysisMode"
    )

    .transform_filter(
        "(SelectedName == 'All') || (datum.preusuel == SelectedName)"
    )

    .transform_calculate(
        value="""
        DisplayMode == 'Count'
        ? datum.nombre
        : datum.percentage
        """
    )

    .mark_text(
        align="left",
        baseline="middle",
        dx=5,
        fontSize=12
    )

    .encode(
        x="annais:Q",
        y="value:Q",
        text="preusuel:N",
        color="preusuel:N"
    )
)

chart = (
    alt.layer(
        period_bands,      # <- fond coloré
        event_lines,
        main_chart,
        points,
        # line_labels,
        event_labels,
    )
    .properties(
        width=1300,
        height=700,
        title={
            "text":
                "Evolution of French First Names (1900–2020)",
            "subtitle": [
                "Explore leaders, trends, growth, decline and stability"
            ]
        }
    )
    .encode(
        x=alt.X(
            scale=alt.Scale(domain=[1900, 2025])
        )
    )
    .configure_view(
        strokeWidth=0
    )
    .configure_axis(
        gridColor="#ECECEC",
        labelFontSize=12,
        titleFontSize=14
    )
    .configure_title(
        fontSize=24,
        subtitleFontSize=13,
        anchor="start"
    )
    .interactive()
)

chart

alt.LayerChart(...)

### Visualization Description

This interactive visualization shows how the popularity of French first names evolved between 1900 and 2020, either in absolute numbers of births or as a percentage of all births in a given year. It allows users to explore and compare historically popular, trend-driven, growing, declining, stable, and event-related names while highlighting potential links between naming patterns and major historical events.

### Questions Answered by the Visualization

**Historical Popularity**

* Which names have been the most popular in France over the period 1900–2020?
* How has the popularity of historically dominant names changed over time?

**Trend-Driven Names** (effet de mode)

* Which names experienced short-lived popularity spikes?
* Which names were strongly associated with specific periods or generations?

**Growing Names**

* Which names have shown sustained growth in recent decades?

**Declining Names**

* Which names were popular in the past but are uncommon today?

**Stable Names**

* Which names have maintained relatively consistent popularity across generations?

**Event-Related Names**

* Do major historical events coincide with changes in the popularity of certain names?
* Do names associated with public figures, athletes, or celebrities experience popularity surges after notable events?

**Comparisons**

* How do different names compare across the same time period?
* Do observed trends change when popularity is measured as percentages rather than absolute birth counts?
* Which names dominate a given period relative to other categories of names?
